<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

# Lab 7 — Part 1: Semantic search with vector embeddings

This notebook builds a small retrieval pipeline: load passages, convert each passage into a dense vector with a pretrained Sentence Transformer, store vectors in DuckDB, retrieve passages close to a query, and assemble the results into a grounded prompt. Similarity helps find relevant text; it does not itself generate or verify an answer.


## Environment setup (uv)

Run these commands in a terminal from the directory containing `pyproject.toml`. The project requires Python 3.11 or newer. The dependency list already includes DuckDB (`duckdb`), so syncing the project installs it along with the other declared packages.

Install `uv` if needed, using the command for your operating system:

- macOS / Linux (Terminal): `curl -LsSf https://astral.sh/uv/install.sh | sh`
- Windows (PowerShell): `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"`

Then, from the directory containing `pyproject.toml`, run:

```bash
# Create/update the project environment and install pyproject.toml dependencies
uv sync

# (Optional) Register the project environment as a Jupyter kernel
uv run python -m ipykernel install --user --name lab7 --display-name "Python (lab7)"

# Start JupyterLab using the project environment
uv run jupyter lab
```

In Jupyter, select the `Python (lab7)` kernel. To install DuckDB explicitly as a project dependency if it is ever removed from `pyproject.toml`, run `uv add duckdb`, then run `uv sync` again. Avoid installing packages into a different Python environment than the notebook kernel.

### What is DuckDB?

DuckDB is an embeddable analytical database: it runs inside your Python process (or as a standalone CLI) and can query data with SQL without requiring a separate database server. In this notebook, the Python `duckdb` package stores passage text and embedding vectors in a local database file, then runs vector-distance queries to retrieve passages similar to a question. The Python package and the DuckDB CLI are separate ways to use DuckDB; installing the CLI is optional.

### Optional: install the DuckDB command-line client

The DuckDB CLI is a separate executable from the Python `duckdb` package. On macOS with Homebrew, install it with `brew install duckdb`, then check it with `duckdb --version`. On other platforms, follow the official installation instructions: [DuckDB CLI installation](https://duckdb.org/docs/stable/clients/cli/overview). The CLI is optional for this notebook; its Python code uses the `duckdb` package installed by `uv sync`.


## 1. Import dependencies

DuckDB stores the passage vectors and supports vector-distance queries. Sentence Transformers creates embeddings, while Hugging Face Datasets loads the sample corpus. `pandas` is used for DataFrame type hints and search results. The model and dataset may be downloaded on first use.


In [ ]:
import duckdb
import pandas as pd

import tqdm.auto as tqdm

from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from datasets.arrow_dataset import Dataset

# pandas is used by DBStore annotations and the DataFrame returned by search.

## 2. Implement the vector store

`DBStore` owns the database connection and embedding model. Its methods create the table/index, embed passages, load data, search by vector distance, and close the connection. HNSW is an approximate nearest-neighbor index intended to speed up vector search on larger collections.


In [ ]:
class DBStore:
    def __init__(self, db_path: str = "./database-p1.db", embedding_size: int = 384) -> None:
        """
        Initialize 

        Args:
            db_path: Path to database file
        """
        self.conn = duckdb.connect(db_path)
        self.embedding_size = embedding_size
        self.model = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dimensions
        self._create_table()
        self._create_index()

    def _create_table(self) -> None:
        """Create the table if it doesn't exist"""
        try:
            # Use IF NOT EXISTS to avoid errors when table already exists
            create_table_query = f"""
            CREATE TABLE IF NOT EXISTS search_engine (
                doc_id UInt32,
                content String,
                embedding FLOAT[{self.embedding_size}]
            )
            """
            self.conn.execute(create_table_query)
        except Exception as e:
            print(f"Warning when creating table: {str(e)}")
            print("Continuing with existing table...")
    
    def _create_index(self) -> None:
        try:
            query = """
            -- Install extension for Vector Similarity Search 
            INSTALL vss;
            -- Load extension
            LOAD vss;
            SET hnsw_enable_experimental_persistence = true;
            -- drop index if exist
            DROP INDEX IF EXISTS idx;
            -- Create an HNSW (Hiearchical Navigable Small World) index on the column 'embedding'
            CREATE INDEX idx ON search_engine USING HNSW (embedding)
            WITH (metric = 'cosine');
            """
            self.conn.execute(query)
        except Exception as e:
            print(f"Warning when creating index: {str(e)}")

    def _generate_embedding(self, df: pd.DataFrame) -> list[list[float]]:
        # generate embedding
        embeddings = self.model.encode(
            df["passage"].tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,   # useful for cosine similarity
        )
        return embeddings.tolist()

    def search(self, query: str, topk: int = 5) -> pd.DataFrame | None:
        try:
            query_embedding = self.model.encode([query], normalize_embeddings=True).tolist()[0]
    
            db_query = f"""
            SELECT 
                doc_id, 
                content,
                array_distance(embedding, {query_embedding}::FLOAT[{self.embedding_size}]) AS distance,
                RANK() OVER (ORDER BY distance) AS rank
            FROM search_engine
            ORDER BY array_distance(embedding, {query_embedding}::FLOAT[{self.embedding_size}])
            LIMIT {topk};
            """
            return self.conn.sql(db_query).df() 
        except Exception as e:
            print(f"Warning when doing query: {str(e)}")
        
    def load_dataset(self, dataset: Dataset) -> None:
        try:
            df = dataset.to_pandas()
            df["embedding"] = self._generate_embedding(df)
            # Load in database
            query ="""
            DELETE FROM search_engine;
            INSERT INTO search_engine (doc_id, content, embedding)
            SELECT 
                id AS doc_id,
                passage AS content,
                embedding
            FROM df
            """
            self.conn.execute(query)
        except Exception as e:
            print(f"Warning when loading dataset: {str(e)}")
    
    def close(self) -> None:
        self.conn.close()

## 3. Load the corpus and questions

The dataset provides Wikipedia passages and a separate question-answer split. This demonstration searches the passages for one manually chosen question.


In [ ]:
# Load the passage corpus to index and the test question-answer split.
corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")["passages"]
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer")["test"]

## 4. Embed and search

The query is embedded with the same model used for passages. The database orders passages by `array_distance`; smaller distances indicate closer vectors for this metric. The connection is closed after retrieval, while the returned results DataFrame remains available.


In [ ]:
# Initialize the vector database, encode the corpus, and search for the question.
db = DBStore()
db.load_dataset(corpus)

# Search passages using the same embedding model used to index them.
question = 'Was Abraham Lincoln the sixteenth President of the United States?'
results = db.search(question)

# Close the connection after retrieval; `results` remains a DataFrame.
db.close()

In [ ]:
results

## 5. Build a context-grounded prompt

The prompt includes the retrieved passages with source indices and instructs a downstream language model to answer only from that context. This notebook constructs and prints the prompt; it does not call a language model to generate an answer.


In [ ]:
def build_prompt(df: pd.DataFrame, question: str) -> str:
    """Format retrieved passages and a question as a grounded prompt.

    Args:
        df: Search results containing a `content` column.
        question: Question to answer using the retrieved context.

    Returns:
        A prompt string containing the context and question.
    """
    # Number passages so their provenance can be referenced.
    context = '\n\n'.join([f"[{idx+1}] {content}" for idx, content in enumerate(df.content)])
    prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say you don't know.
    
Context:
{context}
    
Question: {question}
Answer:
    """
    return prompt

In [ ]:
# Build the prompt from retrieved search results.
prompt = build_prompt(results, question)


In [ ]:
# Inspect the prompt; this notebook does not generate an answer.
print(prompt)


# Student Exercises

Use the `DBStore`, `corpus`, `question`, `results`, and `build_prompt` objects from the preceding tutorial. Rerun the notebook from the beginning before starting. Note that this notebook retrieves passages and formats a prompt; it does not generate an answer with a language model.

## Exercise 1 — Experiment with retrieval depth

Search for the same question with `topk` values of 1, 3, and 10. Compare the returned passages and their distances. Does increasing `topk` always add useful context? Explain your observations.


In [ ]:
# Reopen the store because the main workflow closes its connection after searching.
experiment_db = DBStore()
for k in [1, 3, 10]:
    # TODO: Search using the chosen number of nearest passages.
    retrieved = experiment_db.search(question, topk=k)
    print(f"\nTop {k} results:")
    if retrieved is not None:
        for row in retrieved.itertuples():
            print(f"rank={row.rank}, distance={row.distance:.4f}: {row.content[:180]}...")
experiment_db.close()


## Exercise 2 — Ask a different question

Write two questions about different topics that occur in the corpus. Retrieve the top passages for each. Do the passages support the questions? Compare the rankings and explain one case where a close vector may still provide insufficient evidence.


In [ ]:
# TODO: Replace the examples with questions of your choice.
student_questions = [
    "When was Abraham Lincoln elected president?",
    "What was the capital of Uruguay?",
]

question_db = DBStore()
for student_question in student_questions:
    print(f"\nQuestion: {student_question}")
    retrieved = question_db.search(student_question, topk=3)
    if retrieved is not None:
        for row in retrieved.itertuples():
            print(f"  [{row.rank}] distance={row.distance:.4f}: {row.content[:200]}...")
question_db.close()


## Exercise 3 — Inspect and improve the prompt

Build a prompt from the retrieved passages. Modify the instructions so the answer should cite the passage numbers (for example, `[1]`). Print the updated prompt and check that the numbered context is clear. Why is it useful to tell a language model to say when the context does not contain an answer?


In [ ]:
# Build a prompt using the retrieved results from the previous exercise.
if retrieved is not None:
    citation_prompt = build_prompt(retrieved, student_questions[-1])
    # TODO: Update the instructions to request citations to context passage numbers.
    citation_prompt = citation_prompt.replace(
        "If the answer is not in the context, say you don't know.",
        "Cite supporting passage numbers in square brackets. If the context is insufficient, say you don't know.",
    )
    print(citation_prompt)


## Exercise 4 — Evaluate retrieval quality

For one question, define which retrieved passage IDs you consider relevant. Try several `topk` values and calculate precision@k: the number of relevant results in the first k divided by k. Discuss what precision@k does and does not tell you about retrieval quality.

### Optional challenge

Change the embedding model (keeping its output dimension compatible with the database schema) or compare a semantic search result with a simple keyword baseline. Document the model name, observed results, and limitations.
